## DPO Preference Pair Generation 

### Claude version: Convert RAFT to DPO format

#### Cell 1 — Install (skip if already installed)

In [ ]:
!pip install -q vllm

#### Cell 2 — Imports

In [1]:
import json, os, time
from pathlib import Path
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

#### Cell 3 — Config

In [ ]:
RAFT_FILE  = "/kaggle/input/raft-data/raft_data.jsonl"
OUTPUT_FILE = "/kaggle/working/preference_pairs.jsonl"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"        # swap to whichever you're testing with
MAX_NEW_TOKENS = 120
MAX_MODEL_LEN = 4096    # 2048
CHUNK_SIZE = 2000


In [ ]:
from kaggle_secrets import UserSecretsClient
from kaggle.api.kaggle_api_extended import KaggleApi
import os

user_secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = 'kanav608'
os.environ['KAGGLE_KEY'] = 'alpha_beta'

api = KaggleApi()
api.authenticate()

KAGGLE_DATASET = "kanav608/raft-intermediate"

4096

In [ ]:
# Upload To a Dataset

def upload_to_dataset(dir_path, message="Update DPO files"):
    """
    Commit the given directory as a new version of a Kaggle Dataset.
    Ensures dataset-metadata.json exists before uploading.
    """
    metadata_path = os.path.join(dir_path, "dataset-metadata.json")

    # If metadata file is missing, create a minimal one
    if not os.path.exists(metadata_path):
        # Minimal required metadata – adjust id to your dataset
        metadata = {
            "id": KAGGLE_DATASET,
            "title": "RAFT training examples",
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f)
        print("📄 Created dataset-metadata.json")

    try:
        api.dataset_create_version(
            folder=dir_path,
            version_notes=message,
            quiet=False   # set to True if you want less output
        )
        print(f"Dataset version committed: {message}")
    except Exception as e:
        print(f"Failed to upload: {e}")

#### Cell 4 — Load model with vLLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    gpu_memory_utilization=0.85,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
)
print("vLLM engine loaded.")

#### Cell 5 — Prompt builders + real token-length pre-filter

In [5]:
import hashlib

SAFETY_MARGIN = MAX_NEW_TOKENS + 50

def build_full_prompt(rec):
    docs_block = "\n\n".join(rec["documents"])
    return f"""{rec['instruction']}

{docs_block}

Question: {rec['question']}"""

def build_rejected_gen_prompt(rec):
    docs_block = "\n\n".join(rec["documents"])
    perspective = rec.get("perspective", "")
    historian = rec.get("historian", "")
    hint = f"You are a {perspective} historian ({historian}). " if perspective and historian else ""
    return f"""{hint}Below are some documents and a question about them.

{docs_block}

Question: {rec['question']}

Write a plausible-sounding answer to the question that contains exactly one deliberate factual error (wrong date, wrong person, wrong place, or wrong event), while matching the style and length of a real answer. Output ONLY the answer text — no explanation, no labels, no quotes."""

def fits_context(prompt_text):
    formatted = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}], tokenize=False, add_generation_prompt=True
    )
    n = len(tokenizer(formatted, add_special_tokens=False)["input_ids"])
    return n <= (MAX_MODEL_LEN - SAFETY_MARGIN)

def prompt_hash(prompt_text):
    return hashlib.md5(prompt_text.encode("utf-8")).hexdigest()[:16]

#### Cell 6 — Batched rejected-answer generation via vLLM  + recursive bisection fallback

In [ ]:
def batch_generate_rejected(prompts, temperature=0.8, max_tokens=MAX_NEW_TOKENS):
    formatted = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True
        )
        for p in prompts
    ]
    params = SamplingParams(temperature=temperature, top_p=0.9, max_tokens=max_tokens)
    outputs = llm.generate(formatted, params, use_tqdm=True)
    return [o.outputs[0].text.strip().strip('"') for o in outputs]

def generate_chunk_safe(chunk):
    """Try a chunk; on failure bisect and retry so only the true bad record(s) get dropped."""
    if not chunk:
        return []
    try:
        prompts = [build_rejected_gen_prompt(r) for r in chunk]
        rejected = batch_generate_rejected(prompts)
        return list(zip(chunk, rejected))
    except Exception as e:
        if len(chunk) == 1:
            print(f"Dropping 1 unrecoverable record (historian={chunk[0].get('historian')}): {e}")
            return []
        mid = len(chunk) // 2
        return generate_chunk_safe(chunk[:mid]) + generate_chunk_safe(chunk[mid:])

#### Cell 7 — Main loop: build DPO pairs from existing RAFT (one)file


In [ ]:
with open(RAFT_FILE, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(records)} RAFT records.")

# Figure out what's already done (from your 1,908 successful pairs)
already_done = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                already_done.add(json.loads(line).get("_qid"))
print(f"Already have {len(already_done)} pairs from previous run — will skip those.")

# Real token-length pre-filter, skipping already-done records
remaining = []
dropped_oversized = 0
for r in records:
    fp = build_full_prompt(r)
    qid = prompt_hash(fp)
    if qid in already_done:
        continue
    if not fits_context(build_rejected_gen_prompt(r)):
        dropped_oversized += 1
        continue
    remaining.append((qid, r))

print(f"Dropped {dropped_oversized} oversized records.")
print(f"{len(remaining)} records left to process.\n")

t0 = time.time()
with open(OUTPUT_FILE, "a", encoding="utf-8") as fout:   # append — keeps your 1,908
    for start in range(0, len(remaining), CHUNK_SIZE):
        chunk_items = remaining[start:start + CHUNK_SIZE]
        chunk = [r for _, r in chunk_items]
        print(f"--- Chunk {start}-{start+len(chunk)} / {len(remaining)} ---")

        results = generate_chunk_safe(chunk)

        count = 0
        for rec, rejected in results:
            chosen = rec["output"].strip()
            if not rejected or not chosen or rejected.strip().lower() == chosen.lower():
                continue
            fp = build_full_prompt(rec)
            pair = {
                "prompt": fp,
                "chosen": chosen,
                "rejected": rejected,
                "source": rec.get("historian", "unknown"),
                "_qid": prompt_hash(fp),
            }
            fout.write(json.dumps(pair, ensure_ascii=False) + "\n")
            fout.flush()
            count += 1
        print(f"   → wrote {count} pairs this chunk")

print(f"\nDone in {time.time()-t0:.1f}s.")

In [ ]:
import shutil, time

DPO_DATASET_DIR = "/kaggle/working/dpo_outputs"
os.makedirs(DPO_DATASET_DIR, exist_ok=True)

if os.path.exists(OUTPUT_FILE) and os.path.getsize(OUTPUT_FILE) > 0:
    dest = os.path.join(DPO_DATASET_DIR, os.path.basename(OUTPUT_FILE))
    shutil.copy(OUTPUT_FILE, dest)

    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        total_pairs = sum(1 for _ in f)

    print(f"{total_pairs} pairs found. Uploading to Kaggle dataset...")
    upload_to_dataset(DPO_DATASET_DIR, f"DPO preference pairs — {total_pairs} pairs")
else:
    print("OUTPUT_FILE missing or empty — check Cell 7 for errors before uploading.")

### Data Cleanup

#### Cell 1 — Detect bad rows and match them back to their original RAFT record

In [ ]:
import json, hashlib

RAFT_FILE = "/kaggle/input/raft-data/raft_train.jsonl"
DPO_FILE = "/kaggle/input/datasets/kanav608/raft-intermediate/preference_pairs.jsonl"
FIXED_FILE = "/kaggle/working/preference_pairs_fixed.jsonl"

def prompt_hash(prompt_text):
    return hashlib.md5(prompt_text.encode("utf-8")).hexdigest()[:16]

def build_full_prompt(rec):
    docs_block = "\n\n".join(rec["documents"])
    return f"""{rec['instruction']}

{docs_block}

Question: {rec['question']}"""

def is_meaningfully_different(chosen, rejected):
    def content_words(text):
        return set(w.strip('.,;:()"\'').lower() for w in text.split() if len(w) > 3)
    c_words = content_words(chosen)
    r_words = content_words(rejected)
    if not c_words:
        return True
    overlap = len(c_words & r_words) / len(c_words)
    return overlap < 0.75

# Build qid -> original RAFT record lookup
raft_by_qid = {}
with open(RAFT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        raft_by_qid[prompt_hash(build_full_prompt(rec))] = rec

print(f"Indexed {len(raft_by_qid)} RAFT records.")

good_pairs, bad_pairs = [], []
with open(DPO_FILE, "r", encoding="utf-8") as f:
    for line in f:
        pair = json.loads(line)
        if is_meaningfully_different(pair["chosen"], pair["rejected"]):
            good_pairs.append(pair)
        else:
            bad_pairs.append(pair)

print(f"Good pairs: {len(good_pairs)}")
print(f"Bad pairs (need regeneration): {len(bad_pairs)}")

missing_qid = sum(1 for p in bad_pairs if p.get("_qid") not in raft_by_qid)
print(f"Bad pairs missing a RAFT match: {missing_qid}")

#### Cell 2 — spaCy entity-swap corruption (fast path, no GPU)

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

In [10]:
# !pip install -q spacy
# !python -m spacy download en_core_web_sm -q

import spacy, random
nlp = spacy.load("en_core_web_sm")

DATE_POOL = ["1757", "1857", "1885", "1905", "1919", "1930", "1942", "1947"]
NAME_POOL = ["Gandhi", "Nehru", "Tilak", "Gokhale", "Bose", "Motilal Nehru", "Irwin", "Jinnah"]
PLACE_POOL = ["Bengal", "Bombay", "Punjab", "Madras", "Delhi", "Calcutta"]

def corrupt_answer(answer):
    doc = nlp(answer)
    ents = [e for e in doc.ents if e.label_ in ("DATE", "PERSON", "GPE", "ORG", "LOC")]
    if not ents:
        return None
    target = random.choice(ents)
    if target.label_ == "DATE":
        pool = DATE_POOL
    elif target.label_ == "PERSON":
        pool = NAME_POOL
    else:
        pool = PLACE_POOL
    candidates = [p for p in pool if p.lower() != target.text.lower()]
    if not candidates:
        return None
    replacement = random.choice(candidates)
    return answer[:target.start_char] + replacement + answer[target.end_char:]

####  Cell 3 - LLM fallback for answers with no swappable entity 

In [11]:
from vllm import SamplingParams

def build_rejected_gen_prompt_v2(rec, chosen):
    docs_block = "\n\n".join(rec["documents"])
    return f"""Below are documents and a question about them.

{docs_block}

Question: {rec['question']}

The correct answer is: "{chosen}"

Write a DIFFERENT answer that CONTRADICTS or gets a key fact wrong compared to the correct answer above — not a rewording of it. It must sound plausible and match the style/length of the correct answer, but change what actually happened (wrong date, wrong person, wrong outcome, etc.).
Output ONLY the corrupted answer text."""

def batch_generate_rejected_v2(prompts, temperature=0.9, max_tokens=120):
    formatted = [
        tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
        for p in prompts
    ]
    params = SamplingParams(temperature=temperature, top_p=0.9, max_tokens=max_tokens)
    outputs = llm.generate(formatted, params, use_tqdm=True)
    return [o.outputs[0].text.strip().strip('"') for o in outputs]

#### Cell 4 — Regenerate: entity-swap first, LLM only for the leftovers

In [ ]:
fixed_pairs = []
needs_llm = []   # (pair, rec) tuples

for pair in bad_pairs:
    rec = raft_by_qid.get(pair.get("_qid"))
    if rec is None:
        continue  # can't regenerate without the source record — will report count below

    swapped = corrupt_answer(pair["chosen"])
    if swapped and is_meaningfully_different(pair["chosen"], swapped):
        pair["rejected"] = swapped
        fixed_pairs.append(pair)
    else:
        needs_llm.append((pair, rec))

print(f"Fixed via entity-swap (no GPU): {len(fixed_pairs)}")
print(f"Needs LLM fallback: {len(needs_llm)}")

if needs_llm:
    prompts = [build_rejected_gen_prompt_v2(rec, pair["chosen"]) for pair, rec in needs_llm]
    rejected_v2 = batch_generate_rejected_v2(prompts)

    still_bad = 0
    for (pair, rec), new_rejected in zip(needs_llm, rejected_v2):
        if new_rejected and is_meaningfully_different(pair["chosen"], new_rejected):
            pair["rejected"] = new_rejected
            fixed_pairs.append(pair)
        else:
            still_bad += 1
    print(f"Fixed via LLM: {len(needs_llm) - still_bad}")
    print(f"Still bad after retry (dropped): {still_bad}")

#### Cell 5 — Merge and save

In [ ]:
final_pairs = good_pairs + fixed_pairs

with open(FIXED_FILE, "w", encoding="utf-8") as f:
    for p in final_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"\n🎉 Final dataset: {len(final_pairs)} pairs")
print(f"  - Originally good: {len(good_pairs)}")
print(f"  - Fixed: {len(fixed_pairs)}")
print(f"  - Dropped (unrecoverable): {len(bad_pairs) - len(fixed_pairs)}")
print(f"Saved to: {FIXED_FILE}")

#### Cell 6 —  Clean the data, remove "_uid"

In [14]:
# Fix data , remove _uid
import json

IN_FILE = "/kaggle/working/preference_pairs_fixed.jsonl"
OUT_FILE = "/kaggle/working/preference_pairs_final.jsonl"

with open(IN_FILE, "r", encoding="utf-8") as fin, open(OUT_FILE, "w", encoding="utf-8") as fout:
    for line in fin:
        p = json.loads(line)
        clean = {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
        fout.write(json.dumps(clean, ensure_ascii=False) + "\n")

print("Cleaned file saved to:", OUT_FILE)

Cleaned file saved to: /kaggle/working/preference_pairs_final.jsonl


#### Cell 7 —  Upload data to a dataset 

In [ ]:
import shutil, time

DPO_DATASET_DIR = "/kaggle/working/dpo_outputs"
os.makedirs(DPO_DATASET_DIR, exist_ok=True)

if os.path.exists(FIXED_FILE) and os.path.getsize(FIXED_FILE) > 0:
    dest = os.path.join(DPO_DATASET_DIR, os.path.basename(FIXED_FILE))
    shutil.copy(FIXED_FILE, dest)

    with open(FIXED_FILE, "r", encoding="utf-8") as f:
        total_pairs = sum(1 for _ in f)

    print(f"{total_pairs} pairs found. Uploading to Kaggle dataset...")
    upload_to_dataset(DPO_DATASET_DIR, f"DPO preference pairs — {total_pairs} pairs")
else:
    print("FIXED_FILE missing or empty — check Cell 7 for errors before uploading.")